# Brazilian E-Commerce Data Analysis

This notebook explores the Brazilian e-commerce dataset using **Python, NumPy and Pandas**.

The analysis covers:
- Order trends and delivery performance
- Order-item and seller analysis
- Product data quality
- Payment patterns
- Customer, seller, geolocation and review data
- Basic data cleaning and export of cleaned tables

## 1. Setup

**Before running:** place the original CSV files in the project's `data/raw/` folder. This notebook is in `notebooks/`, so it reads the data using relative paths.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# Set your project directory here
PROJECT_DIR = Path.cwd().parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data folder exists:", RAW_DATA_DIR.exists())
print("Processed data folder exists:", PROCESSED_DATA_DIR.exists())

## 2. Orders Analysis

Load the orders table and inspect its structure, completeness, order statuses and unique order IDs.

In [ ]:
orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")

display(orders.head())
print("Shape:", orders.shape)
orders.info()

In [ ]:
# Check missing values and order-status distribution
display(orders.isnull().sum())
display(orders["order_status"].value_counts())

In [ ]:
# Check which order statuses have missing delivery dates
missing_delivery_by_status = (
    orders[orders["order_delivered_customer_date"].isnull()]["order_status"]
    .value_counts()
)
display(missing_delivery_by_status)

# Check whether any delivered orders are missing the customer delivery date
delivered_missing_date = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].isnull())
]
display(delivered_missing_date)
display(delivered_missing_date.isnull().sum())

In [ ]:
# Check order identifiers and status values
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())
print("Unique order statuses:", orders["order_status"].unique())
print("Number of order statuses:", orders["order_status"].nunique())

### 2.1 Date preparation and order trends

Convert the timestamp columns to datetime values, then derive year and month fields for time-based analysis.

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders.info()

In [ ]:
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_month_name"] = orders["order_purchase_timestamp"].dt.month_name()

display(
    orders[
        ["order_purchase_timestamp", "purchase_year",
         "purchase_month", "purchase_month_name"]
    ].head()
)

In [ ]:
print("Orders by year:")
display(orders["purchase_year"].value_counts().sort_index())

print("Orders by month number:")
display(orders["purchase_month"].value_counts().sort_index())

monthly_orders = (
    orders.groupby(["purchase_year", "purchase_month"])
    .size()
    .reset_index(name="total_orders")
)

display(monthly_orders.head(15))
display(
    monthly_orders.sort_values("total_orders", ascending=False).head(10)
)

In [ ]:
print("First purchase date:", orders["order_purchase_timestamp"].min())
print("Last purchase date:", orders["order_purchase_timestamp"].max())

### 2.2 Delivery-time analysis

Calculate the elapsed time between purchase and customer delivery. Missing delivery dates remain missing and are excluded from summary calculations.

In [ ]:
orders["delivery_time"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
)

orders["delivery_days"] = (
    orders["delivery_time"].dt.total_seconds() / 86400
)

display(
    orders[
        ["order_purchase_timestamp", "order_delivered_customer_date",
         "delivery_time", "delivery_days"]
    ].head(10)
)

print("Average delivery time (days):", orders["delivery_days"].mean())
print("Orders with a calculable delivery time:", orders["delivery_days"].count())

In [ ]:
# Positive values mean delivery happened after the estimated date.
# Negative values mean delivery happened before the estimated date.
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

late_orders = (orders["delivery_delay_days"] > 0).sum()
early_orders = (orders["delivery_delay_days"] < 0).sum()
on_time_orders = (orders["delivery_delay_days"] == 0).sum()
orders_with_delay_data = orders["delivery_delay_days"].notna().sum()

print("Late orders:", late_orders)
print("Early orders:", early_orders)
print("Exactly on estimated date:", on_time_orders)
print("Orders with delivery-delay data:", orders_with_delay_data)

if orders_with_delay_data:
    print(
        "Late delivery percentage:",
        round(late_orders / orders_with_delay_data * 100, 2),
        "%"
    )

print("Average delivery delay (days):", orders["delivery_delay_days"].mean())
display(orders["delivery_delay_days"].describe())

In [ ]:
delay_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay_days",
]

print("Five most delayed orders:")
display(orders.nlargest(5, "delivery_delay_days")[delay_columns])

print("Five orders delivered furthest ahead of estimate:")
display(orders.nsmallest(5, "delivery_delay_days")[delay_columns])

In [ ]:
# Identify statistical outliers in delivery delay using the IQR method.
Q1 = orders["delivery_delay_days"].quantile(0.25)
Q3 = orders["delivery_delay_days"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

outliers = orders[
    (orders["delivery_delay_days"] < lower_bound)
    | (orders["delivery_delay_days"] > upper_bound)
]

print("Number of delivery-delay outliers:", outliers.shape[0])

## 3. Order Items and Seller Analysis

Inspect order-item records, items per order, price and freight totals, and seller-level sales.

In [ ]:
order_items = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")

display(order_items.head())
print("Shape:", order_items.shape)
print("Unique orders in order items:", order_items["order_id"].nunique())

In [ ]:
items_per_order = order_items.groupby("order_id").size()

print("Items per order (sample):")
display(items_per_order.head())

print("Distribution of item counts per order:")
display(items_per_order.value_counts().sort_index())

print("Total item price:", order_items["price"].sum())
print("Total freight value:", order_items["freight_value"].sum())
print("Average item price:", order_items["price"].mean())

In [ ]:
seller_sales = order_items.groupby("seller_id")["price"].sum()

print("Top 10 sellers by item sales:")
display(seller_sales.sort_values(ascending=False).head(10))

print("Number of unique sellers in order items:", order_items["seller_id"].nunique())

## 4. Product Analysis

Inspect product attributes, missing values, duplicates, descriptive statistics and category distribution.

In [ ]:
products = pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv")

display(products.head())
products.info()
display(products.isnull().sum())

In [ ]:
print("Products with missing category:")
display(products[products["product_category_name"].isnull()].head())

print("Products with missing weight:")
display(products[products["product_weight_g"].isnull()])

print("Duplicate product IDs:", products["product_id"].duplicated().sum())
display(products.describe())

In [ ]:
print("Products with zero weight:", (products["product_weight_g"] == 0).sum())

dimension_columns = [
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]
print("Zero values in product dimensions:")
display((products[dimension_columns] == 0).sum())

print("Number of product categories:", products["product_category_name"].nunique())
print("Top 10 product categories:")
display(products["product_category_name"].value_counts().head(10))

print("Completely duplicated product rows:", products.duplicated().sum())
print("Rows containing at least one missing value:", products.isnull().any(axis=1).sum())

## 5. Payment Analysis

Inspect payment records, missing values, duplicates, payment types, installments and payment values.

In [ ]:
payments = pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")

display(payments.head())
payments.info()
display(payments.isnull().sum())

In [ ]:
print("Duplicate payment rows:", payments.duplicated().sum())
print("Repeated order IDs in payment records:", payments["order_id"].duplicated().sum())

print("Payment type counts:")
display(payments["payment_type"].value_counts())

display(payments.describe())

print("Records with zero installments:", (payments["payment_installments"] == 0).sum())
print("Records with zero payment value:", (payments["payment_value"] == 0).sum())

In [ ]:
print("Average payment value by payment type:")
display(payments.groupby("payment_type")["payment_value"].mean())

print("Total payment value by payment type:")
display(
    payments.groupby("payment_type")["payment_value"]
    .sum()
    .sort_values(ascending=False)
)

print("Installment distribution:")
display(payments["payment_installments"].value_counts().sort_index())

## 6. Customer and Seller Analysis

Check the customer and seller tables for structure, duplicate rows and unique identifiers.

In [ ]:
customers = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")

display(customers.head())
customers.info()

print("Duplicate customer rows:", customers.duplicated().sum())
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Unique customer account IDs:", customers["customer_unique_id"].nunique())

In [ ]:
sellers = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")

display(sellers.head())
sellers.info()

print("Duplicate seller rows:", sellers.duplicated().sum())
print("Unique seller IDs:", sellers["seller_id"].nunique())

## 7. Geolocation Analysis

Inspect the geolocation table and identify repeated rows and unique postal-code prefixes.

In [ ]:
geolocation = pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv")

display(geolocation.head())
geolocation.info()

print("Duplicate geolocation rows:", geolocation.duplicated().sum())
print(
    "Unique postal-code prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)
display(geolocation["geolocation_zip_code_prefix"].unique()[:10])

## 8. Review Analysis

Inspect review records, duplicate values and the distribution of review scores.

In [ ]:
reviews = pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset.csv")

display(reviews.head())
reviews.info()

print("Duplicate review rows:", reviews.duplicated().sum())
print("Review-score distribution:")
display(reviews["review_score"].value_counts().sort_index())

## 9. Data Cleaning

Create separate cleaned copies so the original DataFrames remain available for comparison.

Cleaning decisions in this notebook:
- Remove exact duplicate rows from the geolocation table.
- Fill missing product categories with `unknown`.
- Convert review timestamps and the order-item shipping-limit timestamp to datetime.
- Preserve other missing values where they may represent meaningful order states or unavailable information.

In [ ]:
# Geolocation: remove exact duplicate rows.
geolocation_clean = geolocation.drop_duplicates().copy()
print("Geolocation shape before cleaning:", geolocation.shape)
print("Geolocation shape after cleaning:", geolocation_clean.shape)

In [ ]:
# Products: retain rows and label missing categories explicitly.
products_clean = products.copy()
products_clean["product_category_name"] = (
    products_clean["product_category_name"].fillna("unknown")
)

print("Missing product categories after cleaning:",
      products_clean["product_category_name"].isnull().sum())
print("Duplicate product IDs:", products_clean["product_id"].duplicated().sum())
display(products_clean.isnull().sum())

In [ ]:
# Payments: create a separate copy; no rows are removed here.
payments_clean = payments.copy()

print("Rows with zero payment value:")
display(payments_clean[payments_clean["payment_value"] == 0])

print("Rows with zero installments:")
display(payments_clean[payments_clean["payment_installments"] == 0])

In [ ]:
# Reviews: convert review timestamps to datetime.
reviews_clean = reviews.copy()

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

for col in review_date_columns:
    reviews_clean[col] = pd.to_datetime(reviews_clean[col], errors="coerce")

reviews_clean.info()

print("Duplicate review IDs:", reviews_clean["review_id"].duplicated().sum())
display(
    reviews_clean[
        reviews_clean["review_id"].duplicated(keep=False)
    ].sort_values("review_id").head(10)
)
print("Repeated order IDs in reviews:", reviews_clean["order_id"].duplicated().sum())

In [ ]:
# Orders: preserve the analysed data and check data quality.
orders_clean = orders.copy()

display(orders_clean.isnull().sum())
display(orders_clean["order_status"].value_counts())

print("Missing delivery dates by order status:")
display(
    orders_clean[
        orders_clean["order_delivered_customer_date"].isnull()
    ]["order_status"].value_counts()
)

print("Delivered orders missing customer delivery date:")
display(
    orders_clean[
        (orders_clean["order_status"] == "delivered")
        & (orders_clean["order_delivered_customer_date"].isnull())
    ]
)

print("Duplicate order IDs:", orders_clean["order_id"].duplicated().sum())

In [ ]:
# Order items: convert the shipping-limit timestamp and check duplicates.
order_items_clean = order_items.copy()

print("Missing values:")
display(order_items_clean.isnull().sum())

print("Duplicate rows:", order_items_clean.duplicated().sum())

order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce",
)

print("Shipping-limit date dtype:", order_items_clean["shipping_limit_date"].dtype)

In [ ]:
# Customer and seller tables: create copies for a consistent cleaned-data export.
customers_clean = customers.copy()
sellers_clean = sellers.copy()

## 10. Export Cleaned Data

Save the cleaned tables to `data/processed/`. The notebook uses a project-relative output path, so it does not depend on a personal Downloads folder or computer username.

In [ ]:
cleaned_datasets = {
    "orders": orders_clean,
    "order_items": order_items_clean,
    "products": products_clean,
    "payments": payments_clean,
    "customers": customers_clean,
    "sellers": sellers_clean,
    "geolocation": geolocation_clean,
    "reviews": reviews_clean,
}

for name, df in cleaned_datasets.items():
    output_path = PROCESSED_DATA_DIR / f"{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {name}: {df.shape[0]:,} rows, {df.shape[1]} columns -> {output_path}")

## 11. Summary

This notebook documents the exploratory analysis and selected cleaning steps performed on the Brazilian e-commerce dataset. The SQL analysis and Power BI dashboard are maintained separately in the project folders.

**Reproducibility note:** Run the notebook from its own folder (`notebooks/`) or adjust the relative paths if your working directory is different. The original raw dataset files are expected in `data/raw/`.